# Большое домашнее задание 2

In [ ]:
pip install sacrebleu comet-ml

In [ ]:
#!git clone https://github.com/k1ngofdarks/DL-Transformers.git

In [ ]:
#!mv DL-Transformers DL_Transformers
#!ls

In [ ]:
import torch

try:
    from data_utils import create_dataloaders, load_data_splits
    from modeling import TransformerModel
    from train_utils import train_model, translate_and_save
except:
    from DL_Transformers.data_utils import create_dataloaders, load_data_splits
    from DL_Transformers.modeling import TransformerModel
    from DL_Transformers.train_utils import train_model, translate_and_save


In [ ]:
train_de, train_en, val_de, val_en, test_de, data_folder = load_data_splits()
print("Train size:", len(train_de))
print("Val   size:", len(val_de))
print("Test  size:", len(test_de))
print("DATA_FOLDER:", data_folder)


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.float32 if DEVICE.type == "cuda" else torch.float32
print("device:", DEVICE)
print(f"AMP dtype: {amp_dtype}")


In [ ]:
BATCH_SIZE = 128
MAX_LEN = 80


de_tokenizer, en_tokenizer, train_loader, val_loader = create_dataloaders(
    train_de=train_de,
    train_en=train_en,
    val_de=val_de,
    val_en=val_en,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
)


In [ ]:
from comet_ml import Experiment

experiment = Experiment(
    api_key="1kc44ZRsKlVzok3SMdIgY8E6v",
    project_name="dl-transformers"
)

In [ ]:
model = TransformerModel(
    src_vocab=len(de_tokenizer),
    tgt_vocab=len(en_tokenizer),
    pad_id_src=de_tokenizer.word2id["<pad>"],
    pad_id_tgt=en_tokenizer.word2id["<pad>"],
    d_model=256,
    nhead=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=512,
    dropout=0.1,
    max_len=100,
).to(DEVICE)

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    src_tokenizer=de_tokenizer,
    tgt_tokenizer=en_tokenizer,
    val_src_texts=val_de,
    val_tgt_texts=val_en,
    epochs=15,
    device=DEVICE,
    amp_dtype=amp_dtype,
    comet_experiment=experiment,
)


In [ ]:
translate_and_save(
    model=model,
    src_sentences=test_de,
    src_tokenizer=de_tokenizer,
    tgt_tokenizer=en_tokenizer,
    device=DEVICE,
    filename="translation_test1.en",
    batch_size=128,
    comet_experiment=None,
)
